# 📓 Notebook 5 — Pandas Preview: Your First Real DataFrame

> **Module:** Data Science Libraries · **Estimated time:** 25–30 min · **Difficulty:** Beginner

You can do real data work in pure Python — we did exactly that in Notebooks 3 and 4. But once your data is more than a few rows, you want a **proper table**: row + column labels, fast filtering, group-by operations, reading and writing CSV files. That is **pandas**.

Pandas is *the* library data scientists touch every single day. This notebook gives you a working preview — enough to read tabular data and answer basic questions about it. You'll use pandas in every notebook from here on.

## 🎯 Learning objectives

1. Understand the two pandas data structures — **Series** and **DataFrame**.
2. Create DataFrames from dictionaries, lists, and CSV files.
3. Inspect data with `head()`, `tail()`, `info()`, `describe()`, `shape`, `dtypes`.
4. Select **rows** and **columns** by label and by condition.
5. Filter using boolean masks.
6. Add and modify columns.
7. Compute summary statistics and `groupby` aggregations.
8. Make a quick exploratory plot directly from a DataFrame.

## ✅ Prerequisites

Notebooks 1–4 (lists, dictionaries, comprehensions, list of dicts).

## 📦 Installation in Colab

Pandas is preinstalled in Google Colab. If you ever need to install it locally:

```
!pip install pandas
```

## 1. The pandas mental model

Pandas has two core objects:

| Object        | Conceptually          | Compare to                        |
|---------------|-----------------------|-----------------------------------|
| `Series`      | A single column       | a labelled NumPy 1-D array        |
| `DataFrame`   | A whole table         | a labelled 2-D array / SQL table  |

```
   ┌────────────────  DataFrame  ────────────────┐
   │  name       age   score    city            │
   ├─────────────────────────────────────────────┤
0  │  Alice      25    87.5     Berlin          │   ← row 0
1  │  Bob        30    91.0     Munich          │
2  │  Charlie    27    79.2     Hamburg         │
   └─────────────────────────────────────────────┘
       column   column column   column           ← each column is a Series
```

In [ ]:
# The standard import alias — you'll see "pd" in every pandas-using notebook ever
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy  version: {np.__version__}")

## 2. Creating a DataFrame from a dictionary

The most natural way to build a small DataFrame is from a `dict` whose **keys are column names** and **values are lists** (one per row).

In [ ]:
data = {
    "name":  ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "age":   [25, 30, 27, 35, 22, 41],
    "score": [87.5, 91.0, 79.2, 94.3, 68.7, 82.1],
    "city":  ["Berlin", "Munich", "Hamburg", "Berlin", "Munich", "Hamburg"],
}

df = pd.DataFrame(data)
df

**A pandas tip.** Inside Jupyter, just typing the variable name (`df`) at the bottom of a cell renders a beautifully-formatted HTML table. Way nicer than `print(df)` — try both.

## 3. Inspecting a DataFrame

Whenever you load a new dataset, run these four commands first. They tell you the *shape*, the *types*, and the *typical values* before you do anything else.

In [ ]:
print("Shape (rows, cols):", df.shape)
print("\nColumn names :", df.columns.tolist())
print("\nIndex        :", df.index.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# The first / last few rows
print("--- head(3) ---")
print(df.head(3))

print("\n--- tail(2) ---")
print(df.tail(2))

In [ ]:
# Quick statistical summary for numeric columns
df.describe()

In [ ]:
# Schema / non-null counts — VERY useful for spotting missing data
df.info()

## 4. Selecting columns

These three patterns appear in every pandas notebook:

```python
df["score"]              # one column → Series
df[["name", "score"]]    # several columns → DataFrame
df.score                  # attribute-style (only works if column name is a valid identifier)
```

In [ ]:
# One column → Series
scores = df["score"]
print(scores)
print(f"\ntype: {type(scores).__name__}")

In [ ]:
# Several columns → DataFrame
subset = df[["name", "score", "city"]]
subset

## 5. Selecting rows — `loc` vs `iloc`

Pandas gives you **two** ways to slice rows. The difference matters once your DataFrame has a non-default index.

| Indexer | What it uses          | Endpoint inclusive?   |
|---------|------------------------|-----------------------|
| `.loc`  | **labels** (the index) | yes (`df.loc[0:2]` returns 3 rows) |
| `.iloc` | **integer positions**  | no (`df.iloc[0:2]` returns 2 rows) |

In [ ]:
# .loc — label-based
print("df.loc[0]:")
print(df.loc[0])           # first row, returns a Series

print("\ndf.loc[0:2, ['name', 'score']]:")
print(df.loc[0:2, ["name", "score"]])

In [ ]:
# .iloc — position-based
print("df.iloc[0]:")
print(df.iloc[0])

print("\ndf.iloc[0:2, 0:3]:")
print(df.iloc[0:2, 0:3])    # first 2 rows, first 3 cols

> 🎯 **Rule of thumb.** Prefer `.loc` when you can — it reads almost like English. Use `.iloc` only when you really do mean "the *n*-th row" regardless of labels.

## 6. Filtering — the boolean-mask idiom

This is **the** pattern of pandas: build a Series of `True/False` values that says which rows you want, then index the DataFrame with it.

```python
df[ df["score"] > 85 ]
   └─────────────┘
   boolean Series the same length as df
```

In [ ]:
# Step 1: see the mask
mask = df["score"] > 85
print("Mask:")
print(mask)
print()

# Step 2: apply it
print("High scorers:")
print(df[mask])

In [ ]:
# Combining conditions — note the parentheses around each part, and & (not and)
high_and_young = df[(df["score"] > 80) & (df["age"] < 30)]
print(high_and_young)

# OR uses |
in_berlin_or_top = df[(df["city"] == "Berlin") | (df["score"] > 90)]
print("\nBerlin OR top scorer:")
print(in_berlin_or_top)

> ⚠️ Two common stumbles:
> - Use **`&` and `|`**, *not* `and` / `or`, for element-wise boolean operations.
> - **Wrap each condition in parentheses** because `&` and `|` have higher precedence than `>` and `==`.

## 7. Adding and modifying columns

In [ ]:
# Add a column from a formula
df["score_pct"] = df["score"] / 100

# Add a column from a list (same length as df)
df["passed"] = df["score"] >= 70   # boolean column

# A conditional / categorical column
df["grade"] = pd.cut(
    df["score"],
    bins=[0, 60, 70, 80, 90, 100],
    labels=["F", "D", "C", "B", "A"],
)

df

## 8. Summary statistics & `value_counts`

In [ ]:
print("Score stats:")
print(f"  mean   : {df['score'].mean():.2f}")
print(f"  median : {df['score'].median():.2f}")
print(f"  std    : {df['score'].std():.2f}")
print(f"  min    : {df['score'].min():.2f}")
print(f"  max    : {df['score'].max():.2f}")

print("\nHow many people in each city:")
print(df["city"].value_counts())

## 9. `groupby` — the most useful pandas verb

`groupby` answers "what's the *something* of *something_else*, broken down by *category*?". For instance: "average score by city", "total revenue per region", "number of customers per year".

The mental model is **split → apply → combine**:

```
            apply (mean, sum, count, ...)
              ↓
  ┌─ Berlin  : ████  →  87.0
  │
df┼─ Munich  : ███   →  79.8     ← combine into a new DataFrame / Series
  │
  └─ Hamburg : ██    →  80.6
        ↑
      split by the "city" column
```

In [ ]:
# Average score per city
print(df.groupby("city")["score"].mean())

# Several aggregations at once, by city
agg = df.groupby("city").agg(
    n_people   =("name",  "count"),
    mean_score =("score", "mean"),
    mean_age   =("age",   "mean"),
)
agg

## 10. Reading and writing CSV files

This is how every real project starts and ends. Pandas reads dozens of file formats (CSV, Excel, JSON, Parquet, SQL, …) with a one-line call.

In [ ]:
# Reading from a string is great for self-contained demos.
# In a real notebook this would be:  df_sales = pd.read_csv("sales.csv")

from io import StringIO

csv_text = '''
product,sales,region,quarter
Laptop,1200,North,Q1
Phone,800,South,Q1
Tablet,600,East,Q1
Laptop,1350,North,Q2
Phone,900,South,Q2
Tablet,750,East,Q2
Laptop,1500,North,Q3
Phone,950,South,Q3
Tablet,820,East,Q3
'''
df_sales = pd.read_csv(StringIO(csv_text))
df_sales

In [ ]:
# Group-by example on real-ish data: total sales per region
sales_by_region = df_sales.groupby("region")["sales"].sum().sort_values(ascending=False)
print(sales_by_region)

# Quarterly trend
quarterly = df_sales.groupby("quarter")["sales"].sum()
print("\nQuarterly:")
print(quarterly)

## 11. A first plot — `df.plot()`

Pandas has a built-in `.plot()` method that wraps matplotlib. It's perfect for quick exploration.

In [ ]:
import matplotlib.pyplot as plt

# Bar chart of region totals
ax = sales_by_region.plot(
    kind="bar",
    color=["#4C72B0", "#DD8452", "#55A467"],
    title="Total sales by region",
    figsize=(7, 4),
    rot=0,
    edgecolor="black",
)
ax.set_ylabel("Total sales (€)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Quarterly trend per product
pivot = df_sales.pivot_table(index="quarter", columns="product", values="sales", aggfunc="sum")
print(pivot)

pivot.plot(
    marker="o",
    title="Quarterly sales trend per product",
    figsize=(8, 4.5),
    grid=True,
)
plt.ylabel("Sales (€)")
plt.tight_layout()
plt.show()

## 12. The pandas patterns you'll see everywhere

Burn these into memory — they form the daily-use vocabulary of every data scientist:

```python
# Loading
df = pd.read_csv("data.csv")

# Exploring
df.head()                         # see the first rows
df.shape                          # rows, cols
df.describe()                     # summary stats
df.info()                         # types and missing counts

# Selecting
df["col"]                          # one column
df[["col1", "col2"]]               # several columns
df.loc[df["age"] > 30, "name"]     # filter rows + pick a column

# Filtering
df[df["score"] > 85]               # rows above threshold
df[(df["score"] > 85) & (df["city"] == "Berlin")]

# Aggregating
df.groupby("city")["score"].mean()
df.pivot_table(index="quarter", columns="product", values="sales", aggfunc="sum")
```

## 🧪 Practice exercises

For the next exercises we'll use a small "sales transactions" DataFrame.

In [ ]:
rng = np.random.default_rng(42)
n = 50

sales_data = pd.DataFrame({
    "transaction_id": range(1, n + 1),
    "product"       : rng.choice(["Laptop", "Phone", "Tablet", "Watch"], n),
    "region"        : rng.choice(["North", "South", "East", "West"], n),
    "quarter"       : rng.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "amount"        : rng.integers(100, 2000, n).astype(float),
    "customer_age"  : rng.integers(18, 70, n),
})
sales_data.head()

### Exercise 1 — Basic exploration

1. Print the **shape** and **dtypes**.
2. Print the **mean** and **median** of `amount`.
3. Show how many transactions there were in each `region`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
print(f"shape  = {sales_data.shape}")
print(f"\ndtypes:\n{sales_data.dtypes}")

print(f"\nmean amount   = {sales_data['amount'].mean():.2f}")
print(f"median amount = {sales_data['amount'].median():.2f}")

print("\nTransactions per region:")
print(sales_data['region'].value_counts())
```
</details>

### Exercise 2 — Filtering

Find all transactions where the **amount is at least 1500 AND the customer is at least 40 years old**. How many are there? What's their average amount?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
big_old = sales_data[(sales_data["amount"] >= 1500) & (sales_data["customer_age"] >= 40)]

print(f"Matching rows : {len(big_old)}")
print(f"Mean amount   : €{big_old['amount'].mean():,.2f}")
big_old.head()
```
</details>

### Exercise 3 — Group-by

For each `product`, compute:

1. Total sales `amount`.
2. Number of transactions.
3. The region with the highest *total* amount (think: groupby on two columns).

Tip: `df.groupby(["product", "region"])["amount"].sum()` returns a stacked Series — combine with `.idxmax()` or `.sort_values()`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# 1 & 2: per-product summary
summary = sales_data.groupby("product").agg(
    total_sales =("amount", "sum"),
    n_transactions=("amount", "count"),
)
print(summary)

# 3: top region per product
grouped = sales_data.groupby(["product", "region"])["amount"].sum()
print("\nTop region per product:")
print(grouped.groupby(level=0).idxmax())   # tuple (product, region)
```
</details>

### Exercise 4 — Visual exploration

Create two plots:

1. A **bar chart** of total sales per `region`.
2. A **histogram** of `customer_age` with 10 bins.

Add titles and axis labels.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1) Bar chart
sales_data.groupby("region")["amount"].sum().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="black"
)
axes[0].set_title("Total sales per region")
axes[0].set_ylabel("Total sales (€)")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

# 2) Histogram
sales_data["customer_age"].plot(
    kind="hist", bins=10, ax=axes[1], color="#DD8452", edgecolor="black"
)
axes[1].set_title("Customer age distribution")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.show()
```
</details>

### Exercise 5 — Debug me 🐞

The cell below should print the **average amount in each region** but it raises an error. Fix it.

In [ ]:
# Buggy:
print(sales_data.groupby(region).mean(amount))


<details>
<summary>💡 <b>Solution</b></summary>

Two issues:

1. `region` and `amount` need to be **strings** (column names), not bare identifiers.
2. `.mean(amount)` is not how you pick a column — `.mean()` takes no positional column name.

```python
# Correct:
print(sales_data.groupby("region")["amount"].mean())
```

Or, using `.agg`:

```python
print(sales_data.groupby("region").agg(avg=("amount", "mean")))
```
</details>

## 🧠 Key takeaways

1. A **DataFrame** is a labelled table; a **Series** is one column.
2. Use `df.head() / .shape / .dtypes / .info() / .describe()` to inspect a new dataset.
3. Select columns with `df["col"]` or `df[[col1, col2]]`; rows with `.loc[label]` or `.iloc[position]`.
4. **Boolean masks** drive filtering: `df[mask]`. Combine masks with `&`, `|`, `~`, and wrap each condition in parentheses.
5. **`groupby` → aggregation** is the workhorse of analysis (`mean`, `sum`, `count`, custom funcs).
6. `df.plot(...)` is the fast path to exploratory charts.
7. Read CSV with `pd.read_csv("path.csv")`. That single line replaces dozens of pure-Python lines.

## ✅ Self-assessment

- [ ] Build a DataFrame from a `dict` of lists.
- [ ] Select a single column and several columns.
- [ ] Filter rows on one and two conditions.
- [ ] Add a new computed column.
- [ ] Compute the mean / total / count grouped by a category.
- [ ] Render a bar or line chart from a Series or DataFrame.

## 🚀 Next step

Continue with **Notebook 6 — Functions and Modules**, then **Notebook 7 — NumPy Fundamentals**. From there you'll be ready to attack proper visualisation and machine learning.